## PySpark - Instalando a biblioteca PySpark no Google Colab

In [1]:
!pip install pyspark

In [2]:
!pip install findspark

In [4]:
import findspark
findspark.init() # comando de inicialização pyspark
from pyspark.sql import SparkSession # função inicializa a sessão de uso do pyshpark no noteebok
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [5]:
df = spark.sql('''select 'Sucesso total, estamos online!' as hello''')
df.show()

+--------------------+
|               hello|
+--------------------+
|Sucesso total, es...|
+--------------------+



In [6]:
# Import spark libraries
from pyspark.sql import Row, DataFrame
from pyspark.sql.types import StringType, StructType, StructField, IntegerType
from pyspark.sql.functions import col, expr, lit, substring, concat, concat_ws, when, coalesce
from pyspark.sql import functions as F # for more sql functions
from functools import reduce

# Data manipulation using spark

In [11]:
df = spark.read.csv('/banklist.csv', sep = ',', inferSchema = True, header = True) #inferschema = forçar a inferencia quando o arquivo for importado /header informa que tem cabeçalho

print('df.count  :', df.count()) # conta a quantidade de linhas
print('df.col ct :', len(df.columns)) #conta a quantidade de colunas
print('df.columns:', df.columns) # informa o nome das colunas

df.count  : 561
df.col ct : 6
df.columns: ['Bank Name', 'City', 'ST', 'CERT', 'Acquiring Institution', 'Closing Date']


# Using SQL in PySpark

In [13]:
df.createOrReplaceTempView("banklist")

df_check = spark.sql('''select `Bank Name`, City, `Closing Date` from banklist''') # selecionar quais colunas quer trazer
df_check.show(4, truncate=False) # esa clausula é para informar quantas linhas quero que me mosntre

+--------------------------------+-------------+------------+
|Bank Name                       |City         |Closing Date|
+--------------------------------+-------------+------------+
|The First State Bank            |Barboursville|3-Apr-20    |
|Ericson State Bank              |Ericson      |14-Feb-20   |
|City National Bank of New Jersey|Newark       |1-Nov-19    |
|Resolute Bank                   |Maumee       |25-Oct-19   |
+--------------------------------+-------------+------------+
only showing top 4 rows



# Dataframe Basic Operations

In [14]:
df.describe().show() #resumo em alto nível de varias formulas

+-------+--------------------+-------+----+-----------------+---------------------+------------+
|summary|           Bank Name|   City|  ST|             CERT|Acquiring Institution|Closing Date|
+-------+--------------------+-------+----+-----------------+---------------------+------------+
|  count|                 561|    561| 561|              561|                  561|         561|
|   mean|                NULL|   NULL|NULL|31685.68449197861|                 NULL|        NULL|
| stddev|                NULL|   NULL|NULL|16446.65659309965|                 NULL|        NULL|
|    min|1st American Stat...|Acworth|  AL|               91|      1st United Bank|    1-Aug-08|
|    max|               ebank|Wyoming|  WY|            58701|  Your Community Bank|    9-Sep-11|
+-------+--------------------+-------+----+-----------------+---------------------+------------+



In [15]:
df.describe('City', 'ST').show() # colunas especifica

+-------+-------+----+
|summary|   City|  ST|
+-------+-------+----+
|  count|    561| 561|
|   mean|   NULL|NULL|
| stddev|   NULL|NULL|
|    min|Acworth|  AL|
|    max|Wyoming|  WY|
+-------+-------+----+



# Counts, Columns and Schema

Schema é a estrutura de organização de dados — uma espécie de “mapa” que descreve como os dados estão organizados dentro de um banco, tabela ou dataset.

Em bancos de dados relacionais (SQL):
O schema define tabelas, colunas, tipos de dados e relacionamentos.

In [16]:
print('Total de linhas:', df.count()) # contar linhas
print('Total de colunas:', len(df.columns)) # contar colunas
print('Tipo de dados:', df.dtypes) # contar tipos de dados
print('Schema:', df.schema) #mosntar o schema

Total de linhas: 561
Total de colunas: 6
Tipo de dados: [('Bank Name', 'string'), ('City', 'string'), ('ST', 'string'), ('CERT', 'int'), ('Acquiring Institution', 'string'), ('Closing Date', 'string')]
Schema: StructType([StructField('Bank Name', StringType(), True), StructField('City', StringType(), True), StructField('ST', StringType(), True), StructField('CERT', IntegerType(), True), StructField('Acquiring Institution', StringType(), True), StructField('Closing Date', StringType(), True)])


In [18]:
print('df schema 1:')
df.printSchema() # para ver o conteudo desse schema dos dados

df schema 1:
root
 |-- Bank Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- ST: string (nullable = true)
 |-- CERT: integer (nullable = true)
 |-- Acquiring Institution: string (nullable = true)
 |-- Closing Date: string (nullable = true)



# Remove Duplicates

In [23]:
df = df.dropDuplicates() # retirar as duplicidades
print('df.count		:', df.count()) # contar novamente linhas para validar se tinha duplicidade e comparar com os dadpos anterior
print('df.columns	:', df.columns) # contar novamente colunas ""

df.count		: 561
df.columns	: ['Bank Name', 'City', 'ST', 'CERT', 'Acquiring Institution', 'Closing Date']


# Select specific columns

In [24]:
df2 = df.select(*['Bank Name', 'City']) # seleciona somente colunas específicas
df2.show()

+--------------------+----------------+
|           Bank Name|            City|
+--------------------+----------------+
| First Bank of Idaho|         Ketchum|
|Amcore Bank, Nati...|        Rockford|
|        Venture Bank|           Lacey|
|First State Bank ...|           Altus|
|Valley Capital Ba...|            Mesa|
|Michigan Heritage...|Farmington Hills|
|Columbia Savings ...|      Cincinnati|
|       Fidelity Bank|        Dearborn|
|The Park Avenue Bank|        Valdosta|
|Western Commercia...|  Woodland Hills|
|        Syringa Bank|           Boise|
|Republic Federal ...|           Miami|
|Westside Communit...|University Place|
|   First United Bank|           Crete|
|HarVest Bank of M...|    Gaithersburg|
|            BankEast|       Knoxville|
|    Polk County Bank|        Johnston|
|Colorado Capital ...|     Castle Rock|
|         Access Bank|        Champlin|
|Pacific National ...|   San Francisco|
+--------------------+----------------+
only showing top 20 rows



# Select multiple columns

In [33]:
col_l = list(set(df.columns)  - {'CERT','ST'}) # Atrubir todas as colunas exceto as colunas sinalizadas
df2 = df.select(*col_l)
df2.show()

+--------------------+---------------------+----------------+------------+
|           Bank Name|Acquiring Institution|            City|Closing Date|
+--------------------+---------------------+----------------+------------+
| First Bank of Idaho|      U.S. Bank, N.A.|         Ketchum|   24-Apr-09|
|Amcore Bank, Nati...|          Harris N.A.|        Rockford|   23-Apr-10|
|        Venture Bank| First-Citizens Ba...|           Lacey|   11-Sep-09|
|First State Bank ...|         Herring Bank|           Altus|   31-Jul-09|
|Valley Capital Ba...| Enterprise Bank &...|            Mesa|   11-Dec-09|
|Michigan Heritage...|       Level One Bank|Farmington Hills|   24-Apr-09|
|Columbia Savings ...| United Fidelity B...|      Cincinnati|   23-May-14|
|       Fidelity Bank| The Huntington Na...|        Dearborn|   30-Mar-12|
|The Park Avenue Bank|   Bank of the Ozarks|        Valdosta|   29-Apr-11|
|Western Commercia...| First California ...|  Woodland Hills|    5-Nov-10|
|        Syringa Bank|   

# Rename columns

In [34]:
#Renomeação de colunas

df2 = df \
  .withColumnRenamed('Bank Name'            , 'bank_name') \
  .withColumnRenamed('Acquiring Institution', 'acq_institution') \
  .withColumnRenamed('Closing Date'         , 'closing_date') \
  .withColumnRenamed('ST'                   , 'state') \
  .withColumnRenamed('CERT'                 , 'cert') #\

df2.show()

SyntaxError: unexpected character after line continuation character (ipython-input-1108588605.py, line 4)

# Add columns

In [35]:
df2 = df.withColumn('state', col('ST')) # adicionar colunas
df2.show()

+--------------------+----------------+---+-----+---------------------+------------+-----+
|           Bank Name|            City| ST| CERT|Acquiring Institution|Closing Date|state|
+--------------------+----------------+---+-----+---------------------+------------+-----+
| First Bank of Idaho|         Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|   ID|
|Amcore Bank, Nati...|        Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|   IL|
|        Venture Bank|           Lacey| WA|22868| First-Citizens Ba...|   11-Sep-09|   WA|
|First State Bank ...|           Altus| OK| 9873|         Herring Bank|   31-Jul-09|   OK|
|Valley Capital Ba...|            Mesa| AZ|58399| Enterprise Bank &...|   11-Dec-09|   AZ|
|Michigan Heritage...|Farmington Hills| MI|34369|       Level One Bank|   24-Apr-09|   MI|
|Columbia Savings ...|      Cincinnati| OH|32284| United Fidelity B...|   23-May-14|   OH|
|       Fidelity Bank|        Dearborn| MI|33883| The Huntington Na...|   30-Mar-12|   MI|

# Add constant column

In [36]:
df2 = df.withColumn('country', lit('US')) # adicionar uma coluna com informação constante
df2.show(2)

+--------------------+--------+---+-----+---------------------+------------+-------+
|           Bank Name|    City| ST| CERT|Acquiring Institution|Closing Date|country|
+--------------------+--------+---+-----+---------------------+------------+-------+
| First Bank of Idaho| Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|     US|
|Amcore Bank, Nati...|Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|     US|
+--------------------+--------+---+-----+---------------------+------------+-------+
only showing top 2 rows



# Drop columns

In [37]:
df2 = df.drop('CERT') #apagar/ dropar a coluna sinalizada
df2.show(2)

+--------------------+--------+---+---------------------+------------+
|           Bank Name|    City| ST|Acquiring Institution|Closing Date|
+--------------------+--------+---+---------------------+------------+
| First Bank of Idaho| Ketchum| ID|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford| IL|          Harris N.A.|   23-Apr-10|
+--------------------+--------+---+---------------------+------------+
only showing top 2 rows



# Drop multiple columns

In [38]:
df2 = df.drop(*['CERT','ST']) # apagar multiplas colunas, passar 2 ou mais colunas para exclusão
df2.show(3)

+--------------------+--------+---------------------+------------+
|           Bank Name|    City|Acquiring Institution|Closing Date|
+--------------------+--------+---------------------+------------+
| First Bank of Idaho| Ketchum|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford|          Harris N.A.|   23-Apr-10|
|        Venture Bank|   Lacey| First-Citizens Ba...|   11-Sep-09|
+--------------------+--------+---------------------+------------+
only showing top 3 rows



In [43]:
df2 = reduce(DataFrame.drop, ['CERT','ST'], df) # exclui as colunas sinalizadas
df2.show(6) # quantidades de linhas a exibir

+--------------------+----------------+---------------------+------------+
|           Bank Name|            City|Acquiring Institution|Closing Date|
+--------------------+----------------+---------------------+------------+
| First Bank of Idaho|         Ketchum|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|        Rockford|          Harris N.A.|   23-Apr-10|
|        Venture Bank|           Lacey| First-Citizens Ba...|   11-Sep-09|
|First State Bank ...|           Altus|         Herring Bank|   31-Jul-09|
|Valley Capital Ba...|            Mesa| Enterprise Bank &...|   11-Dec-09|
|Michigan Heritage...|Farmington Hills|       Level One Bank|   24-Apr-09|
+--------------------+----------------+---------------------+------------+
only showing top 6 rows



# Filter data

In [45]:
# faz referencia ao filtro de uma coluna especifica dentro do data frame
df2 = df.where(df['ST'] == 'NE')

# entre valores
df3 = df.where(df['CERT'].between('1000','2000'))

# possibilidade de passar mais de um valor em uma mesma consulta
df4 = df.where(df['ST'].isin('NE','IL'))

print('df.count  :', df.count()) # filtro geral do arquivo original para comparativo
print('df2.count :', df2.count()) # filtro de valores especificos
print('df3.count :', df3.count()) # filtro entre valores
print('df4.count :', df4.count()) # filtro multiplo


df.count  : 561
df2.count : 4
df3.count : 9
df4.count : 73


# Filter data using logical operators

In [46]:
df2 = df.where((df['ST'] == 'NE') & (df['City'] == 'Ericson')) # possibilidade de executar filtros com operadores logicos , no caso são multiplos filtros um em ST e outro em City, podendo haver mais se necessário
df2.show(3)

+------------------+-------+---+-----+---------------------+------------+
|         Bank Name|   City| ST| CERT|Acquiring Institution|Closing Date|
+------------------+-------+---+-----+---------------------+------------+
|Ericson State Bank|Ericson| NE|18265| Farmers and Merch...|   14-Feb-20|
+------------------+-------+---+-----+---------------------+------------+



# Replace values in dataframe

In [48]:
# Pre replace
df.show(2) # visualizar antes, pré replace

# Post replace
print('Replace 7 in the above dataframe with 17 at all instances')
df.na.replace(7,17).show(2) # alterar o valor 7 para o valor 17

+--------------------+--------+---+-----+---------------------+------------+
|           Bank Name|    City| ST| CERT|Acquiring Institution|Closing Date|
+--------------------+--------+---+-----+---------------------+------------+
| First Bank of Idaho| Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|
+--------------------+--------+---+-----+---------------------+------------+
only showing top 2 rows

Replace 7 in the above dataframe with 17 at all instances
+--------------------+--------+---+-----+---------------------+------------+
|           Bank Name|    City| ST| CERT|Acquiring Institution|Closing Date|
+--------------------+--------+---+-----+---------------------+------------+
| First Bank of Idaho| Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|
+--------------------+--------+---+-----+---------------------+-------